In [1]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
def secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return None

for name in ["WANDB_API_KEY", "HF_TOKEN"]:
    if value := secret(name):
        os.environ[name] = value
if "WANDB_API_KEY" not in os.environ:  # run pushed via the Kaggle CLI has no Secrets: log offline, `wandb sync` later
    os.environ["WANDB_MODE"] = "offline"

REPO_DIR = "/kaggle/working/ai500-labs"
BRANCH = "main"
token = secret("GITHUB_TOKEN")
url = f"https://{token}@github.com/PavloKhomenko/ai500-labs.git" if token else "https://github.com/PavloKhomenko/ai500-labs.git"

In [2]:
# the token is passed only on the command line, origin keeps the public url (/kaggle/working gets saved)
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q {url} {BRANCH}
else:
    !git clone -q -b {BRANCH} {url} {REPO_DIR}
    !git -C {REPO_DIR} remote set-url origin https://github.com/PavloKhomenko/ai500-labs.git
!git -C {REPO_DIR} log --oneline -1

d410670 (HEAD -> main, origin/main, origin/HEAD) fix and results


In [3]:
# versions from lab-1/uv.lock; torch stays Kaggle's CUDA build
!pip install -q "transformers==5.18.0" "datasets==5.1.0" "accelerate==1.15.0" "peft==0.21.2" \
    "huggingface-hub==1.33.0" "wandb==0.30.0" "tabulate==0.10.0" "pyyaml==6.0.3" "sacrebleu==2.6.0"
# Kaggle ships torchao 0.10; peft 0.21 needs torchao > 0.16 and errors while dispatching LoRA layers, and we don't use it
!pip uninstall -q -y torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.1/26.1 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import sys
sys.path.insert(0, f"{REPO_DIR}/lab-1/src")

import copy
import glob
import json
import shutil
from pathlib import Path

import pandas as pd
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

from lab1.common import load_config, out_dir
from lab1.part2 import wandb_log as wl
from lab1.part3.mixlora import Adapter
from lab1.part3.sft import Part3Report, SFTTrainer, SpivavtorData
%matplotlib inline

cfg = load_config()
p3 = cfg["part3"]
device = torch.device("cuda")
env = {"torch": torch.__version__, "transformers": transformers.__version__, "gpu": torch.cuda.get_device_name()}
print(env)
p3_dir = out_dir(cfg, "part3")

def start_run(name, job_type, **extra):
    return wl.init_run(cfg, name, job_type, config={**p3, **env, **extra}, group="part3-v2",
                       project=cfg["wandb"]["part3_project"])

def load_base():
    return AutoModelForCausalLM.from_pretrained(p3["model"], dtype=torch.float16).to(device)

found = glob.glob("/kaggle/input/**/mixlora-none/adapter.pt", recursive=True)
assert found, "add the lab1-part3 notebook output as an Input"
prev = Path(found[0]).parents[1]
print("previous runs:", sorted(d.name for d in prev.iterdir() if (d / "adapter.pt").exists()))

{'torch': '2.11.0+cu128', 'transformers': '5.18.0', 'gpu': 'Tesla T4'}
previous runs: ['lora-r102', 'lora-r16', 'lora-r29', 'mixlora-aux', 'mixlora-bias', 'mixlora-none']


## Дані: train/val як у першому прогоні, тест розширено

In [5]:
tok = AutoTokenizer.from_pretrained(p3["model"])
data = SpivavtorData(tok, p3, seed=cfg["seed"])
data.check_masking()
data.check_disjoint()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

train.jsonl: downloading bytes:           |  0.00B            

val.jsonl:   0%|          | 0.00/3.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/62784 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6976 [00:00<?, ? examples/s]

{'train': 4000, 'val': 300, 'test': 1476}
masking ok: 'І завжди у мене була якась “велика ціль”, на яку я відкладав гроші протягом місяців, а іноді навіть років.<|im_end|>'
splits disjoint: {'train': 3962, 'val': 300, 'test': 1470}


In [6]:
p3_g = copy.deepcopy(p3)
p3_g["bias_update_rate"] = 1e-2

# name: (make adapter, seed, retrain?)
RUNS = {
    "mixlora-none": (lambda m: Adapter.mixlora(m, p3, "none"), 42, False),
    "mixlora-aux": (lambda m: Adapter.mixlora(m, p3, "aux"), 42, False),
    "mixlora-bias": (lambda m: Adapter.mixlora(m, p3, "bias"), 42, False),
    "lora-r16": (lambda m: Adapter.lora(m, 16), 42, False),
    "lora-r29": (lambda m: Adapter.lora(m, 29), 42, False),
    "lora-r102": (lambda m: Adapter.lora(m, 102), 42, False),
    "mixlora-bias-g1e-2": (lambda m: Adapter.mixlora(m, p3_g, "bias"), 42, True),
    "mixlora-none-s43": (lambda m: Adapter.mixlora(m, p3, "none"), 43, True),
    "lora-r102-s43": (lambda m: Adapter.lora(m, 102), 43, True),
}

In [7]:
if not (p3_dir / "base" / "metrics.json").exists():
    start_run("base", "eval")
    trainer = SFTTrainer(Adapter.lora(load_base(), 16), data, p3, seed=cfg["seed"])
    trainer.evaluate(out_dir(cfg, "part3", "base"), {"trainable": 0, "active": 0})
    wl.finish()
    del trainer
    torch.cuda.empty_cache()

wandb: Tracking run with wandb version 0.30.0
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /kaggle/working/wandb/offline-run-20261013_184743-vo9yawh2
wandb: View this run in the terminal with `wandb leet`


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

wandb: 
wandb: Run summary:
wandb:      active 0
wandb:        chrf 54.88961
wandb: exact_match 0.01084
wandb:         f05 0.01723
wandb:   precision 0.01474
wandb:      recall 0.05331
wandb:   trainable 0
wandb:    val_loss 0.40339
wandb: 
wandb: You can sync this run to the cloud by running:
wandb: wandb sync /kaggle/working/wandb/offline-run-20261013_184743-vo9yawh2
wandb: Find logs at: ./wandb/offline-run-20261013_184743-vo9yawh2/logs


{'val_loss': 0.4033941819560515, 'f05': 0.01723235319780124, 'precision': 0.014738805970149253, 'recall': 0.053306342780026994, 'chrf': 54.88960516359892, 'exact_match': 0.01084010840108401, 'trainable': 0, 'active': 0}


In [8]:
for name, (make, seed, retrain) in RUNS.items():
    run_dir = out_dir(cfg, "part3", name)
    if (run_dir / "metrics.json").exists():
        continue
    start_run(name, "train" if retrain else "eval", run=name, seed=seed)
    set_seed(seed)
    trainer = SFTTrainer(make(load_base()), data, p3, seed=seed)
    if retrain:
        timing = trainer.train(run_dir)
        trainer.adapter.save(run_dir / "adapter.pt")
    else:
        trainer.adapter.load(prev / name / "adapter.pt")
        shutil.copy(prev / name / "train_log.jsonl", run_dir / "train_log.jsonl")
        old = json.loads((prev / name / "metrics.json").read_text())
        timing = {k: old[k] for k in ("steps", "sec_per_step", "peak_mem_gb")}
    trainer.evaluate(run_dir, {**timing, "seed": seed})
    wl.finish()
    del trainer
    torch.cuda.empty_cache()

wandb: Tracking run with wandb version 0.30.0
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /kaggle/working/wandb/offline-run-20261013_190541-tn7ftzpv
wandb: View this run in the terminal with `wandb leet`


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

wandb: 
wandb: Run history:
wandb: val/maxvio/layer_00 ▁
wandb: val/maxvio/layer_01 ▁
wandb: val/maxvio/layer_02 ▁
wandb: val/maxvio/layer_03 ▁
wandb: val/maxvio/layer_04 ▁
wandb: val/maxvio/layer_05 ▁
wandb: val/maxvio/layer_06 ▁
wandb: val/maxvio/layer_07 ▁
wandb: val/maxvio/layer_08 ▁
wandb: val/maxvio/layer_09 ▁
wandb:                 +18 ...
wandb: 
wandb: Run summary:
wandb:       active 32915456
wandb:         chrf 93.01282
wandb:  exact_match 0.44038
wandb:          f05 0.40598
wandb:  peak_mem_gb 7.64359
wandb:    precision 0.46503
wandb:       recall 0.26923
wandb: sec_per_step 4.37477
wandb:         seed 42
wandb:        steps 250
wandb:          +30 ...
wandb: 
wandb: You can sync this run to the cloud by running:
wandb: wandb sync /kaggle/working/wandb/offline-run-20261013_190541-tn7ftzpv
wandb: Find logs at: ./wandb/offline-run-20261013_190541-tn7ftzpv/logs


{'val_loss': 0.17527449178992285, 'f05': 0.405982905982906, 'precision': 0.46503496503496505, 'recall': 0.2692307692307692, 'chrf': 93.01282443082467, 'exact_match': 0.44037940379403795, 'trainable': 117555200, 'active': 32915456, 'steps': 250, 'sec_per_step': 4.374765920639038, 'peak_mem_gb': 7.643589496612549, 'seed': 42}


wandb: Tracking run with wandb version 0.30.0
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /kaggle/working/wandb/offline-run-20261013_193231-ozxavvzt
wandb: View this run in the terminal with `wandb leet`


...

wandb: Tracking run with wandb version 0.30.0
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /kaggle/working/wandb/offline-run-20261013_223418-oca4jqia
wandb: View this run in the terminal with `wandb leet`


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{"loss": 0.3681, "aux_per_layer": 0.0, "grad_norm": 4.549, "step": 1, "lr": 2e-05, "sec_per_step": 2.425, "max_mem_gb": 5.107}
{"loss": 0.1786, "aux_per_layer": 0.0, "grad_norm": 0.693, "step": 11, "lr": 0.00012, "sec_per_step": 2.775, "max_mem_gb": 5.401}
{"loss": 0.1734, "aux_per_layer": 0.0, "grad_norm": 1.975, "step": 21, "lr": 0.0002, "sec_per_step": 3.025, "max_mem_gb": 5.5}
{"loss": 0.1485, "aux_per_layer": 0.0, "grad_norm": 0.5178, "step": 31, "lr": 0.000199, "sec_per_step": 2.876, "max_mem_gb": 5.5}
{"loss": 0.1339, "aux_per_layer": 0.0, "grad_norm": 0.4117, "step": 41, "lr": 0.0001963, "sec_per_step": 2.597, "max_mem_gb": 5.5}
{"loss": 0.2384, "aux_per_layer": 0.0, "grad_norm": 0.911, "step": 51, "lr": 0.0001921, "sec_per_step": 2.5, "max_mem_gb": 5.5}
{"loss": 0.4115, "aux_per_layer": 0.0, "grad_norm": 0.8479, "step": 61, "lr": 0.0001863, "sec_per_step": 3.024, "max_mem_gb": 5.5}
{"loss": 0.1001, "aux_per_layer": 0.0, "grad_norm": 0.5037, "step": 71, "lr": 0.000179, "sec_per

wandb: 
wandb: Run history:
wandb: train/aux_per_layer ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:     train/grad_norm █▄▄▂▄▂▁▂▂▂▁▂▂▂▂▁▂▁▂▁▂▂▂▂▂▂▂▁▁▁▁▁▁▂▂▂▁▁▁▁
wandb:          train/loss ▆█▃▅▄▅▄█▇▃▂▁▂▁▄▂▅▂▁▃▃▂▂▂▂▂▃▁▃▅▄▃▂▂▁▅▄▁▂▂
wandb:            train/lr ▇███████▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
wandb:    train/max_mem_gb ▁▁▂▂▄▄▄█████████████████████████████████
wandb:  train/sec_per_step ▁▅▄▅▅▇▂▅▅▆▇▅▆▃▆▂▅▄▂▆▄▅▄▇▁▃▃▄▄▆▄▅▄▆█▅▃▆▃▅
wandb: 
wandb: Run summary:
wandb:       active 117712896
wandb:         chrf 92.31691
wandb:  exact_match 0.41192
wandb:          f05 0.37436
wandb:  peak_mem_gb 5.77748
wandb:    precision 0.40018
wandb:       recall 0.29757
wandb: sec_per_step 2.76582
wandb:         seed 43
wandb:        steps 250
wandb:           +8 ...
wandb: 
wandb: You can sync this run to the cloud by running:
wandb: wandb sync /kaggle/working/wandb/offline-run-20261013_223418-oca4jqia
wandb: Find logs at: ./wandb/offline-run-20261013_223418-oca4jqia/logs


{'val_loss': 0.173010165552669, 'f05': 0.37436332767402375, 'precision': 0.40018148820326677, 'recall': 0.2975708502024291, 'chrf': 92.31690867936358, 'exact_match': 0.41192411924119243, 'trainable': 117712896, 'active': 117712896, 'steps': 250, 'sec_per_step': 2.765815613221149, 'peak_mem_gb': 5.777483940124512, 'seed': 43}


## Результати
Головна метрика: edit F0.5 з 95% bootstrap CI і різницею з `lora-r102`. chrF / EM — допоміжні; рядок `copy` = вихід без змін.

In [9]:
report = Part3Report(p3_dir, ["base", *RUNS])
start_run("summary-v2", "analysis")
table = report.table()
report.routing_figures()
report.loss_curves()
wl.finish()
pd.set_option("display.max_columns", 30)
table.round(4)

wandb: Tracking run with wandb version 0.30.0
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /kaggle/working/wandb/offline-run-20261013_225642-lqqdt0xb
wandb: View this run in the terminal with `wandb leet`


test: 1476 examples, 801 need a correction


wandb: You can sync this run to the cloud by running:
wandb: wandb sync /kaggle/working/wandb/offline-run-20261013_225642-lqqdt0xb
wandb: Find logs at: ./wandb/offline-run-20261013_225642-lqqdt0xb/logs


,run,f05,precision,recall,f05_lo,f05_hi,d_f05_vs_lora-r102,d_lo,d_hi,chrF,chrF_changed,EM,EM_changed,val_loss,trainable_M,active_M,sec_per_step,peak_mem_gb,maxvio_mean
0,base,0.0172,0.0147,0.0533,0.0131,0.0214,-0.3968,-0.4301,-0.3662,54.8896,54.6281,0.0108,0.0012,0.4034,0.0000,0.0000,NaN,NaN,NaN
1,mixlora-none,0.4060,0.4650,0.2692,0.3754,0.4401,-0.0085,-0.0287,0.0123,93.0128,89.9044,0.4404,0.1498,0.1753,117.5552,32.9155,4.3748,7.6436,1.7366
2,mixlora-aux,0.3910,0.4412,0.2686,0.3596,0.4247,-0.0237,-0.0440,-0.0049,92.8087,89.6333,0.4411,0.1498,0.1747,117.5552,32.9155,4.2883,7.6164,0.0543
3,mixlora-bias,0.3991,0.4514,0.2726,0.3669,0.4317,-0.0156,-0.0346,0.0041,92.8777,89.7482,0.4431,0.1536,0.1753,117.5552,32.9155,4.3305,7.6132,0.9473
4,lora-r16,0.3920,0.4471,0.2625,0.3591,0.4235,-0.0227,-0.0409,-0.0042,92.9096,89.6740,0.4336,0.1436,0.1773,18.4648,18.4648,2.3620,4.3419,NaN
5,lora-r29,0.4046,0.4642,0.2672,0.3719,0.4378,-0.0100,-0.0294,0.0101,92.9730,89.7443,0.4478,0.1498,0.1750,33.4674,33.4674,2.4747,4.5655,NaN
6,lora-r102,0.4152,0.4667,0.2881,0.3825,0.4481,0.0000,0.0000,0.0000,93.0896,89.9096,0.4465,0.1610,0.1709,117.7129,117.7129,2.5895,5.8295,NaN
7,mixlora-bias-g1e-2,0.3983,0.4545,0.2665,0.3660,0.4328,-0.0161,-0.0364,0.0036,92.9256,89.7726,0.4383,0.1423,0.1752,117.5552,32.9155,4.1817,7.5420,0.0598
8,mixlora-none-s43,0.3748,0.4135,0.2726,0.3417,0.4071,-0.0403,-0.0596,-0.0203,92.6144,89.4711,0.4167,0.1323,0.1752,117.5552,32.9155,4.2551,7.5675,1.6829
9,lora-r102-s43,0.3744,0.4002,0.2976,0.3443,0.4071,-0.0404,-0.0601,-0.0226,92.3169,89.1492,0.4119,0.1336,0.1730,117.7129,117.7129,2.7658,5.7775,NaN


In [10]:
# seed spread for the key pair
table.set_index("run").loc[["mixlora-none", "mixlora-none-s43", "lora-r102", "lora-r102-s43"],
                           ["f05", "f05_lo", "f05_hi", "precision", "recall", "val_loss"]].round(4)

,f05,f05_lo,f05_hi,precision,recall,val_loss
run,,,,,,
mixlora-none,0.4060,0.3754,0.4401,0.4650,0.2692,0.1753
mixlora-none-s43,0.3748,0.3417,0.4071,0.4135,0.2726,0.1752
lora-r102,0.4152,0.3825,0.4481,0.4667,0.2881,0.1709
lora-r102-s43,0.3744,0.3443,0.4071,0.4002,0.2976,0.1730
